In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_uscf import lps_solver

In [2]:
csv_file = 'open_shell_10atoms_vs_uhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 150 rows found.


In [6]:
ATOMS = {
    'H':  {'mult': 2}, 
    'He':  {'mult': 1}, 
    'Li':  {'mult': 2}, 
    'Be': {'mult': 1},
    'B':  {'mult': 2},
    'C':  {'mult': 3},  
    'N': {'mult': 4}, 
    'O': {'mult': 3},
    'F':  {'mult': 2}, 
    'Ne':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'reference': 'UHF',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    rhf_energies[atom] = round(E, 6)
    print(f"UHF/UGBS {atom} Energy: {E:.4f} Hartree")

UHF/UGBS H Energy: -0.5000 Hartree
UHF/UGBS He Energy: -2.8617 Hartree
UHF/UGBS Li Energy: -7.4328 Hartree
UHF/UGBS Be Energy: -14.5730 Hartree
UHF/UGBS B Energy: -24.5293 Hartree
UHF/UGBS C Energy: -37.6900 Hartree
UHF/UGBS N Energy: -54.4045 Hartree
UHF/UGBS O Energy: -74.8140 Hartree
UHF/UGBS F Energy: -99.4114 Hartree
UHF/UGBS Ne Energy: -128.5471 Hartree


In [9]:
atom_order = ['H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['UHF/UGBS'] = pd.Series(rhf_energies)
# new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
# energy_table = energy_table[new_order]
energy_table

Method,TF0.111111W FA,TF0.111111W LDA,TF0.111111W PBE,TF0.166666W FA,TF0.166666W LDA,TF0.166666W PBE,TF0.2W FA,TF0.2W LDA,TF0.2W PBE,TF0.333333W FA,TF0.333333W LDA,TF0.333333W PBE,TFW FA,TFW LDA,TFW PBE,UHF/UGBS
Atom,,,,,,,,,,,,,,,,
H,-0.535383,-0.507896,-0.552329,-0.496992,-0.468770,-0.509366,-0.477757,-0.449396,-0.488070,-0.416983,-0.389402,-0.422357,-0.263808,-0.243294,-0.263417,-0.500000
He,-3.284537,-3.222808,-3.383279,-3.019522,-2.951216,-3.097076,-2.889052,-2.818308,-2.957074,-2.489526,-2.414978,-2.532515,-1.539225,-1.477450,-1.547985,-2.861680
Li,-8.130332,-8.223685,-8.485256,-7.523259,-7.599164,-7.838804,-7.223393,-7.291246,-7.520201,-6.299982,-6.345962,-6.542940,-4.055442,-4.070028,-4.194061,-7.432751
Be,-15.745337,-16.163144,-16.541291,-14.660505,-15.040403,-15.388855,-14.123257,-14.484084,-14.818155,-12.460449,-12.762758,-13.053328,-8.318528,-8.492186,-8.681694,-14.573023
B,-26.465664,-27.267375,-27.760995,-24.741194,-25.489132,-25.945773,-23.884296,-24.604654,-25.043286,-21.218778,-21.851561,-22.235876,-14.472785,-14.888970,-15.145418,-24.529314
C,-40.537668,-41.792781,-42.403790,-38.015202,-39.201764,-39.768461,-36.758013,-37.909078,-38.454146,-32.829916,-33.866228,-34.346109,-22.753780,-23.487434,-23.812886,-37.689992
N,-58.144747,-59.931895,-60.662545,-54.666523,-56.370353,-57.049296,-52.928699,-54.589045,-55.242778,-47.478821,-48.997021,-49.574636,-33.341834,-34.468237,-34.864735,-54.404541
O,-79.901408,-82.488493,-83.356251,-75.339269,-77.815387,-78.623782,-73.056822,-75.474070,-76.253537,-65.882385,-68.102884,-68.794890,-47.074203,-48.724465,-49.207223,-74.814042
F,-105.588562,-109.044032,-110.047836,-99.779315,-103.099210,-104.035946,-96.868871,-100.116184,-101.020225,-87.700287,-90.701767,-91.506981,-63.469677,-65.732724,-66.300760,-99.411352


In [10]:
reference = energy_table['UHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'UHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)

In [11]:
display(energy_table)

,TF0.111111W FA,TF0.111111W LDA,TF0.111111W PBE,TF0.166666W FA,TF0.166666W LDA,TF0.166666W PBE,TF0.2W FA,TF0.2W LDA,TF0.2W PBE,TF0.333333W FA,TF0.333333W LDA,TF0.333333W PBE,TFW FA,TFW LDA,TFW PBE,UHF/UGBS
H,-0.535383,-0.507896,-0.552329,-0.496992,-0.468770,-0.509366,-0.477757,-0.449396,-0.488070,-0.416983,-0.389402,-0.422357,-0.263808,-0.243294,-0.263417,-0.500000
He,-3.284537,-3.222808,-3.383279,-3.019522,-2.951216,-3.097076,-2.889052,-2.818308,-2.957074,-2.489526,-2.414978,-2.532515,-1.539225,-1.477450,-1.547985,-2.861680
Li,-8.130332,-8.223685,-8.485256,-7.523259,-7.599164,-7.838804,-7.223393,-7.291246,-7.520201,-6.299982,-6.345962,-6.542940,-4.055442,-4.070028,-4.194061,-7.432751
Be,-15.745337,-16.163144,-16.541291,-14.660505,-15.040403,-15.388855,-14.123257,-14.484084,-14.818155,-12.460449,-12.762758,-13.053328,-8.318528,-8.492186,-8.681694,-14.573023
B,-26.465664,-27.267375,-27.760995,-24.741194,-25.489132,-25.945773,-23.884296,-24.604654,-25.043286,-21.218778,-21.851561,-22.235876,-14.472785,-14.888970,-15.145418,-24.529314
C,-40.537668,-41.792781,-42.403790,-38.015202,-39.201764,-39.768461,-36.758013,-37.909078,-38.454146,-32.829916,-33.866228,-34.346109,-22.753780,-23.487434,-23.812886,-37.689992
N,-58.144747,-59.931895,-60.662545,-54.666523,-56.370353,-57.049296,-52.928699,-54.589045,-55.242778,-47.478821,-48.997021,-49.574636,-33.341834,-34.468237,-34.864735,-54.404541
O,-79.901408,-82.488493,-83.356251,-75.339269,-77.815387,-78.623782,-73.056822,-75.474070,-76.253537,-65.882385,-68.102884,-68.794890,-47.074203,-48.724465,-49.207223,-74.814042
F,-105.588562,-109.044032,-110.047836,-99.779315,-103.099210,-104.035946,-96.868871,-100.116184,-101.020225,-87.700287,-90.701767,-91.506981,-63.469677,-65.732724,-66.300760,-99.411352
Ne,-135.511318,-139.886664,-141.026313,-128.291707,-132.508856,-133.573716,-124.669607,-128.801566,-129.829979,-113.234462,-117.075419,-117.993679,-82.790109,-85.734446,-86.387928,-128.547083


In [49]:
csv_file = 'open_shell_15atoms_vs_uhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 175 rows found.


In [13]:
ATOMS = {
    'Na':  {'mult': 2}, 
    'Mg':  {'mult': 1}, 
    'Al':  {'mult': 2}, 
    'Si': {'mult': 3},
    'P':  {'mult': 4},
    'S':  {'mult': 3},  
    'Cl': {'mult': 2}, 
    'Ar': {'mult': 1},
    'K':  {'mult': 2}, 
    'Ca':  {'mult': 1},
    'Zn': {'mult': 1},
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'reference': 'UHF',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    rhf_energies[atom] = round(E, 6)
    print(f"UHF/UGBS {atom} Energy: {E:.4f} Hartree")

UHF/UGBS Na Energy: -161.8589 Hartree
UHF/UGBS Mg Energy: -199.6146 Hartree
UHF/UGBS Al Energy: -241.8768 Hartree
UHF/UGBS Si Energy: -288.8546 Hartree
UHF/UGBS P Energy: -340.7193 Hartree
UHF/UGBS S Energy: -397.5065 Hartree
UHF/UGBS Cl Energy: -459.4829 Hartree
UHF/UGBS Ar Energy: -526.8175 Hartree
UHF/UGBS K Energy: -599.1648 Hartree
UHF/UGBS Ca Energy: -676.7582 Hartree
UHF/UGBS Zn Energy: -1777.8481 Hartree
UHF/UGBS Kr Energy: -2752.0549 Hartree


In [50]:
atom_order = ['Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['UHF/UGBS'] = pd.Series(rhf_energies)
energy_table

Method,TF0.111111W FA,TF0.111111W LDA,TF0.111111W PBE,TF0.166666W FA,TF0.166666W LDA,TF0.166666W PBE,TF0.2W FA,TF0.2W LDA,TF0.2W PBE,TF0.333333W FA,TF0.333333W LDA,TF0.333333W PBE,TFW FA,TFW LDA,TFW PBE,UHF/UGBS
Atom,,,,,,,,,,,,,,,,
Na,-169.876758,-175.220390,-176.494488,-161.083133,-166.247018,-167.439022,-156.665459,-161.732209,-162.884191,-142.690045,-147.423156,-148.454126,-105.226002,-108.912083,-109.651463,-161.858942
Mg,-208.826727,-215.265035,-216.678698,-198.305386,-204.540681,-205.864548,-193.013833,-199.138984,-200.419079,-176.244867,-181.989422,-183.137203,-131.025067,-135.553609,-136.382297,-199.614621
Al,-252.536872,-260.117429,-261.669232,-240.123724,-247.478726,-248.933421,-233.874058,-241.106354,-242.513713,-214.036249,-220.842857,-222.107058,-160.241930,-165.669941,-166.588302,-241.876778
Si,-301.135691,-309.909768,-311.599918,-286.667497,-295.193987,-296.779753,-279.375910,-287.767440,-289.302316,-256.195211,-264.117075,-265.498027,-193.008025,-199.392286,-200.400917,-288.854610
P,-354.731372,-364.755320,-366.584858,-338.045378,-347.799950,-349.517759,-329.628312,-339.235807,-340.899147,-302.831533,-311.925851,-313.424434,-229.434658,-236.833553,-237.933265,-340.719264
S,-413.577377,-425.088095,-427.065091,-394.530559,-405.745598,-407.603230,-384.915848,-395.968957,-397.768400,-354.272546,-364.758190,-366.381630,-270.014427,-278.601238,-279.798321,-397.506544
Cl,-477.678247,NaN,NaN,-456.106275,-468.830862,-470.828673,-445.209435,-457.757749,-459.693620,-410.442846,-422.371240,-424.119790,-314.492584,-324.324636,-325.619280,-459.482916
Ar,-547.190764,-561.816447,-564.088988,-522.928926,-537.208760,-539.346529,-510.665243,-524.754841,-526.826944,-471.497581,-484.916246,-486.789800,-363.016580,-374.146432,-375.538832,-526.817486
Ca,-702.938291,-720.921710,-723.490150,-672.808485,-690.396265,-692.814970,-657.560358,-674.929867,-677.275620,-608.769789,-625.366030,-627.491222,-472.757119,-486.677236,-488.267505,-676.758154


In [51]:
reference = energy_table['UHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'UHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)
display(energy_table)

,TF0.111111W FA,TF0.111111W LDA,TF0.111111W PBE,TF0.166666W FA,TF0.166666W LDA,TF0.166666W PBE,TF0.2W FA,TF0.2W LDA,TF0.2W PBE,TF0.333333W FA,TF0.333333W LDA,TF0.333333W PBE,TFW FA,TFW LDA,TFW PBE,UHF/UGBS
Na,-169.876758,-175.220390,-176.494488,-161.083133,-166.247018,-167.439022,-156.665459,-161.732209,-162.884191,-142.690045,-147.423156,-148.454126,-105.226002,-108.912083,-109.651463,-161.858942
Mg,-208.826727,-215.265035,-216.678698,-198.305386,-204.540681,-205.864548,-193.013833,-199.138984,-200.419079,-176.244867,-181.989422,-183.137203,-131.025067,-135.553609,-136.382297,-199.614621
Al,-252.536872,-260.117429,-261.669232,-240.123724,-247.478726,-248.933421,-233.874058,-241.106354,-242.513713,-214.036249,-220.842857,-222.107058,-160.241930,-165.669941,-166.588302,-241.876778
Si,-301.135691,-309.909768,-311.599918,-286.667497,-295.193987,-296.779753,-279.375910,-287.767440,-289.302316,-256.195211,-264.117075,-265.498027,-193.008025,-199.392286,-200.400917,-288.854610
P,-354.731372,-364.755320,-366.584858,-338.045378,-347.799950,-349.517759,-329.628312,-339.235807,-340.899147,-302.831533,-311.925851,-313.424434,-229.434658,-236.833553,-237.933265,-340.719264
S,-413.577377,-425.088095,-427.065091,-394.530559,-405.745598,-407.603230,-384.915848,-395.968957,-397.768400,-354.272546,-364.758190,-366.381630,-270.014427,-278.601238,-279.798321,-397.506544
Cl,-477.678247,NaN,NaN,-456.106275,-468.830862,-470.828673,-445.209435,-457.757749,-459.693620,-410.442846,-422.371240,-424.119790,-314.492584,-324.324636,-325.619280,-459.482916
Ar,-547.190764,-561.816447,-564.088988,-522.928926,-537.208760,-539.346529,-510.665243,-524.754841,-526.826944,-471.497581,-484.916246,-486.789800,-363.016580,-374.146432,-375.538832,-526.817486
Ca,-702.938291,-720.921710,-723.490150,-672.808485,-690.396265,-692.814970,-657.560358,-674.929867,-677.275620,-608.769789,-625.366030,-627.491222,-472.757119,-486.677236,-488.267505,-676.758154
Zn,-1842.940240,-1882.061216,-1886.161085,-1773.727671,-1812.192488,-1816.067936,-1738.551858,-1776.651097,-1780.417260,-1625.252536,-1662.037293,-1665.473198,-1302.039741,-1334.052914,-1336.686536,-1777.848060


In [47]:
1 / 2 * 3 ** (1 / 3) * np.pi ** (4 / 3)

3.3180041088822585